In [1]:
""

''

In [2]:
import asyncio
from pathlib import Path

# Create the AI predictor module
ai_predictor_code = '''"""
Motor de IA para Previsão de Mercado
Implementa modelos de Machine Learning para prever movimentos de mercado
"""

import asyncio
import numpy as np
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime, timedelta
import joblib
import json
from pathlib import Path

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Deep Learning (LSTM)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False

from ..utils.logger import setup_logger
from ..core.config import Config

logger = setup_logger(__name__, Config.LOG_FILE, Config.LOG_LEVEL)


class TechnicalIndicators:
    """Calcula indicadores técnicos para análise de mercado"""
    
    @staticmethod
    def calculate_rsi(prices: np.ndarray, period: int = 14) -> float:
        """
        Calcula o RSI (Relative Strength Index)
        
        Args:
            prices: Array de preços
            period: Período para cálculo
            
        Returns:
            Valor do RSI (0-100)
        """
        if len(prices) < period + 1:
            return 50.0
        
        deltas = np.diff(prices)
        gains = np.where(deltas > 0, deltas, 0)
        losses = np.where(deltas < 0, -deltas, 0)
        
        avg_gain = np.mean(gains[-period:])
        avg_loss = np.mean(losses[-period:])
        
        if avg_loss == 0:
            return 100.0
        
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
        
        return float(rsi)
    
    @staticmethod
    def calculate_macd(prices: np.ndarray, fast: int = 12, slow: int = 26, signal: int = 9) -> Dict[str, float]:
        """
        Calcula o MACD (Moving Average Convergence Divergence)
        
        Args:
            prices: Array de preços
            fast: Período EMA rápida
            slow: Período EMA lenta
            signal: Período da linha de sinal
            
        Returns:
            Dict com macd, signal e histogram
        """
        if len(prices) < slow:
            return {"macd": 0.0, "signal": 0.0, "histogram": 0.0}
        
        # Calcular EMAs
        ema_fast = TechnicalIndicators._ema(prices, fast)
        ema_slow = TechnicalIndicators._ema(prices, slow)
        
        macd_line = ema_fast - ema_slow
        signal_line = TechnicalIndicators._ema(np.array([macd_line]), signal)
        histogram = macd_line - signal_line
        
        return {
            "macd": float(macd_line),
            "signal": float(signal_line),
            "histogram": float(histogram)
        }
    
    @staticmethod
    def calculate_bollinger_bands(prices: np.ndarray, period: int = 20, std_dev: int = 2) -> Dict[str, float]:
        """
        Calcula as Bandas de Bollinger
        
        Args:
            prices: Array de preços
            period: Período para média móvel
            std_dev: Número de desvios padrão
            
        Returns:
            Dict com upper, middle e lower bands
        """
        if len(prices) < period:
            middle = float(np.mean(prices))
            return {"upper": middle, "middle": middle, "lower": middle}
        
        middle_band = np.mean(prices[-period:])
        std = np.std(prices[-period:])
        
        upper_band = middle_band + (std_dev * std)
        lower_band = middle_band - (std_dev * std)
        
        return {
            "upper": float(upper_band),
            "middle": float(middle_band),
            "lower": float(lower_band)
        }
    
    @staticmethod
    def calculate_moving_averages(prices: np.ndarray) -> Dict[str, float]:
        """
        Calcula médias móveis simples
        
        Args:
            prices: Array de preços
            
        Returns:
            Dict com MA de diferentes períodos
        """
        mas = {}
        periods = [5, 10, 20, 50, 100, 200]
        
        for period in periods:
            if len(prices) >= period:
                mas[f"ma_{period}"] = float(np.mean(prices[-period:]))
            else:
                mas[f"ma_{period}"] = float(np.mean(prices))
        
        return mas
    
    @staticmethod
    def calculate_volume_indicators(volumes: np.ndarray, prices: np.ndarray) -> Dict[str, float]:
        """
        Calcula indicadores baseados em volume
        
        Args:
            volumes: Array de volumes
            prices: Array de preços
            
        Returns:
            Dict com indicadores de volume
        """
        if len(volumes) < 20:
            return {
                "volume_ma": float(np.mean(volumes)),
                "volume_ratio": 1.0,
                "obv": 0.0
            }
        
        volume_ma = np.mean(volumes[-20:])
        current_volume = volumes[-1]
        volume_ratio = current_volume / volume_ma if volume_ma > 0 else 1.0
        
        # OBV (On-Balance Volume)
        obv = 0.0
        for i in range(1, len(prices)):
            if prices[i] > prices[i-1]:
                obv += volumes[i]
            elif prices[i] < prices[i-1]:
                obv -= volumes[i]
        
        return {
            "volume_ma": float(volume_ma),
            "volume_ratio": float(volume_ratio),
            "obv": float(obv)
        }
    
    @staticmethod
    def _ema(prices: np.ndarray, period: int) -> float:
        """Calcula EMA (Exponential Moving Average)"""
        if len(prices) < period:
            return float(np.mean(prices))
        
        multiplier = 2 / (period + 1)
        ema = np.mean(prices[:period])
        
        for price in prices[period:]:
            ema = (price * multiplier) + (ema * (1 - multiplier))
        
        return float(ema)


class AIPredictor:
    """Motor de IA para previsão de mercado usando Machine Learning"""
    
    def __init__(self, model_dir: str = "backend/data/models"):
        """
        Inicializa o preditor de IA
        
        Args:
            model_dir: Diretório para salvar modelos
        """
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)
        
        self.rf_model: Optional[RandomForestClassifier] = None
        self.lstm_model: Optional[Any] = None
        self.scaler: Optional[StandardScaler] = None
        self.feature_names: List[str] = []
        
        self.indicators = TechnicalIndicators()
        self._is_trained = False
        self._last_training = None
        
        logger.info("AIPredictor inicializado")
    
    async def prepare_features(self, candles: List[Dict]) -> Tuple[np.ndarray, List[str]]:
        """
        Prepara features a partir dos candles
        
        Args:
            candles: Lista de candles com OHLCV
            
        Returns:
            Tuple com (features array, feature names)
        """
        if len(candles) < 200:
            logger.warning(f"Poucos candles para análise: {len(candles)}")
        
        # Extrair dados
        closes = np.array([c['close'] for c in candles])
        opens = np.array([c['open'] for c in candles])
        highs = np.array([c['max'] for c in candles])
        lows = np.array([c['min'] for c in candles])
        volumes = np.array([c['volume'] for c in candles])
        
        # Calcular indicadores
        features = {}
        
        # RSI
        features['rsi'] = self.indicators.calculate_rsi(closes)
        
        # MACD
        macd = self.indicators.calculate_macd(closes)
        features.update(macd)
        
        # Bollinger Bands
        bb = self.indicators.calculate_bollinger_bands(closes)
        features['bb_upper'] = bb['upper']
        features['bb_middle'] = bb['middle']
        features['bb_lower'] = bb['lower']
        features['bb_position'] = (closes[-1] - bb['lower']) / (bb['upper'] - bb['lower']) if bb['upper'] != bb['lower'] else 0.5
        
        # Moving Averages
        mas = self.indicators.calculate_moving_averages(closes)
        features.update(mas)
        
        # Volume
        vol_indicators = self.indicators.calculate_volume_indicators(volumes, closes)
        features.update(vol_indicators)
        
        # Price action
        features['price_change'] = (closes[-1] - closes[-2]) / closes[-2] if len(closes) > 1 else 0
        features['price_range'] = (highs[-1] - lows[-1]) / closes[-1] if closes[-1] != 0 else 0
        features['body_size'] = abs(closes[-1] - opens[-1]) / closes[-1] if closes[-1] != 0 else 0
        
        # Momentum
        if len(closes) >= 10:
            features['momentum_10'] = (closes[-1] - closes[-10]) / closes[-10]
        else:
            features['momentum_10'] = 0
        
        # Volatility
        if len(closes) >= 20:
            features['volatility'] = np.std(closes[-20:]) / np.mean(closes[-20:])
        else:
            features['volatility'] = 0
        
        # Trend
        if len(closes) >= 50:
            features['trend_50'] = 1 if closes[-1] > np.mean(closes[-50:]) else -1
        else:
            features['trend_50'] = 0
        
        feature_names = list(features.keys())
        feature_array = np.array([list(features.values())])
        
        return feature_array, feature_names
    
    async def train_random_forest(self, historical_data: List[Dict], labels: np.ndarray) -> Dict[str, float]:
        """
        Treina modelo Random Forest
        
        Args:
            historical_data: Dados históricos de candles
            labels: Labels (1=BUY, 0=HOLD, -1=SELL)
            
        Returns:
            Métricas de performance
        """
        logger.info("Iniciando treinamento do Random Forest...")
        
        # Preparar features
        all_features = []
        for candles in historical_data:
            features, feature_names = await self.prepare_features(candles)
            all_features.append(features[0])
        
        X = np.array(all_features)
        y = labels
        
        # Normalizar
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Treinar modelo
        self.rf_model = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        
        loop = asyncio.get_event_loop()
        await loop.run_in_executor(None, self.rf_model.fit, X_train, y_train)
        
        # Avaliar
        y_pred = self.rf_model.predict(X_test)
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
        }
        
        self.feature_names = feature_names
        self._is_trained = True
        self._last_training = datetime.now()
        
        logger.info(f"Random Forest treinado - Accuracy: {metrics['accuracy']:.4f}")
        
        # Salvar modelo
        await self.save_model('random_forest')
        
        return metrics
    
    async def train_lstm(self, historical_data: List[Dict], labels: np.ndarray, epochs: int = 50) -> Dict[str, float]:
        """
        Treina modelo LSTM
        
        Args:
            historical_data: Dados históricos de candles
            labels: Labels (1=BUY, 0=HOLD, -1=SELL)
            epochs: Número de épocas
            
        Returns:
            Métricas de performance
        """
        if not TENSORFLOW_AVAILABLE:
            logger.warning("TensorFlow não disponível. Usando apenas Random Forest.")
            return {}
        
        logger.info("Iniciando treinamento do LSTM...")
        
        # Preparar sequências
        sequence_length = 60
        all_features = []
        
        for candles in historical_data:
            features, _ = await self.prepare_features(candles)
            all_features.append(features[0])
        
        X = np.array(all_features)
        
        # Normalizar
        if self.scaler is None:
            self.scaler = StandardScaler()
            X_scaled = self.scaler.fit_transform(X)
        else:
            X_scaled = self.scaler.transform(X)
        
        # Criar sequências
        X_seq = []
        y_seq = []
        
        for i in range(sequence_length, len(X_scaled)):
            X_seq.append(X_scaled[i-sequence_length:i])
            y_seq.append(labels[i])
        
        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)
        
        # Converter labels para one-hot
        y_seq_categorical = keras.utils.to_categorical(y_seq + 1, num_classes=3)  # -1,0,1 -> 0,1,2
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_seq, y_seq_categorical, test_size=0.2, random_state=42
        )
        
        # Construir modelo
        self.lstm_model = Sequential([
            LSTM(128, return_sequences=True, input_shape=(sequence_length, X.shape[1])),
            Dropout(0.3),
            LSTM(64, return_sequences=True),
            Dropout(0.3),
            LSTM(32),
            Dropout(0.2),
            Dense(64, activation='relu'),
            Dropout(0.2),
            Dense(3, activation='softmax')
        ])
        
        self.lstm_model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Treinar
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        loop = asyncio.get_event_loop()
        history = await loop.run_in_executor(
            None,
            lambda: self.lstm_model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=32,
                validation_split=0.2,
                callbacks=[early_stop],
                verbose=0
            )
        )
        
        # Avaliar
        y_pred = self.lstm_model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_test_classes = np.argmax(y_test, axis=1)
        
        metrics = {
            'accuracy': accuracy_score(y_test_classes, y_pred_classes),
            'precision': precision_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0),
            'recall': recall_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0),
            'f1': f1_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0)
        }
        
        logger.info(f"LSTM treinado - Accuracy: {metrics['accuracy']:.4f}")
        
        # Salvar modelo
        await self.save_model('lstm')
        
        return metrics
    
    async def predict(self, candles: List[Dict], use_lstm: bool = False) -> Dict[str, Any]:
        """
        Gera previsão de mercado
        
        Args:
            candles: Lista de candles recentes
            use_lstm: Se True, usa LSTM; caso contrário, Random Forest
            
        Returns:
            Dict com signal, confidence e indicators
        """
        if not self._is_trained:
            logger.warning("Modelo não treinado. Carregando modelo salvo...")
            await self.load_model('random_forest')
        
        # Preparar features
        features, feature_names = await self.prepare_features(candles)
        
        # Normalizar
        if self.scaler:
            features_scaled = self.scaler.transform(features)
        else:
            features_scaled = features
        
        # Prever
        if use_lstm and self.lstm_model and TENSORFLOW_AVAILABLE:
            # LSTM precisa de sequência
            prediction_proba = self.lstm_model.predict(features_scaled.reshape(1, 1, -1), verbose=0)[0]
            prediction = np.argmax(prediction_proba) - 1  # 0,1,2 -> -1,0,1
            confidence = float(np.max(prediction_proba))
        elif self.rf_model:
            prediction = self.rf_model.predict(features_scaled)[0]
            prediction_proba = self.rf_model.predict_proba(features_scaled)[0]
            confidence = float(np.max(prediction_proba))
        else:
            logger.error("Nenhum modelo disponível")
            return {
                "signal": "HOLD",
                "confidence": 0.0,
                "indicators": {}
            }
        
        # Mapear sinal
        signal_map = {-1: "SELL", 0: "HOLD", 1: "BUY"}
        signal = signal_map.get(int(prediction), "HOLD")
        
        # Preparar indicadores
        indicators = dict(zip(feature_names, features[0]))
        
        result = {
            "signal": signal,
            "confidence": confidence,
            "indicators": indicators,
            "timestamp": datetime.now().isoformat(),
            "model_used": "LSTM" if use_lstm else "RandomForest"
        }
        
        logger.info(f"Previsão: {signal} (confiança: {confidence:.2%})")
        
        return result
    
    async def save_model(self, model_type: str = 'random_forest') -> None:
        """Salva modelo treinado"""
        try:
            if model_type == 'random_forest' and self.rf_model:
                model_path = self.model_dir / 'rf_model.joblib'
                scaler_path = self.model_dir / 'scaler.joblib'
                
                joblib.dump(self.rf_model, model_path)
                joblib.dump(self.scaler, scaler_path)
                
                # Salvar metadados
                metadata = {
                    'feature_names': self.feature_names,
                    'last_training': self._last_training.isoformat() if self._last_training else None,
                    'model_type': 'RandomForest'
                }
                
                with open(self.model_dir / 'metadata.json', 'w') as f:
                    json.dump(metadata, f, indent=2)
                
                logger.info(f"Modelo Random Forest salvo em {model_path}")
            
            elif model_type == 'lstm' and self.lstm_model and TENSORFLOW_AVAILABLE:
                model_path = self.model_dir / 'lstm_model.h5'
                self.lstm_model.save(model_path)
                logger.info(f"Modelo LSTM salvo em {model_path}")
        
        except Exception as e:
            logger.error(f"Erro ao salvar modelo: {str(e)}")
    
    async def load_model(self, model_type: str = 'random_forest') -> bool:
        """Carrega modelo salvo"""
        try:
            if model_type == 'random_forest':
                model_path = self.model_dir / 'rf_model.joblib'
                scaler_path = self.model_dir / 'scaler.joblib'
                metadata_path = self.model_dir / 'metadata.json'
                
                if not model_path.exists():
                    logger.warning("Modelo Random Forest não encontrado")
                    return False
                
                self.rf_model = joblib.load(model_path)
                self.scaler = joblib.load(scaler_path)
                
                if metadata_path.exists():
                    with open(metadata_path, 'r') as f:
                        metadata = json.load(f)
                        self.feature_names = metadata.get('feature_names', [])
                        self._last_training = datetime.fromisoformat(metadata['last_training']) if metadata.get('last_training') else None
                
                self._is_trained = True
                logger.info("Modelo Random Forest carregado com sucesso")
                return True
            
            elif model_type == 'lstm' and TENSORFLOW_AVAILABLE:
                model_path = self.model_dir / 'lstm_model.h5'
                
                if not model_path.exists():
                    logger.warning("Modelo LSTM não encontrado")
                    return False
                
                self.lstm_model = load_model(model_path)
                logger.info("Modelo LSTM carregado com sucesso")
                return True
        
        except Exception as e:
            logger.error(f"Erro ao carregar modelo: {str(e)}")
            return False
    
    async def should_retrain(self, days_threshold: int = 7) -> bool:
        """
        Verifica se modelo deve ser retreinado
        
        Args:
            days_threshold: Dias desde último treinamento
            
        Returns:
            True se deve retreinar
        """
        if not self._last_training:
            return True
        
        days_since_training = (datetime.now() - self._last_training).days
        return days_since_training >= days_threshold
    
    @property
    def is_trained(self) -> bool:
        """Verifica se modelo está treinado"""
        return self._is_trained
'''

# Write the AI predictor module
await Editor.write('/workspace/backend/services/ai_predictor.py', ai_predictor_code)
print("✓ AI Predictor module created at /workspace/backend/services/ai_predictor.py")

NameError: name 'Editor' is not defined

In [3]:
# Import the Editor tool
from metagpt.tools.libs.editor import Editor

# Create the AI predictor module
ai_predictor_code = '''"""
Motor de IA para Previsão de Mercado
Implementa modelos de Machine Learning para prever movimentos de mercado
"""

import asyncio
import numpy as np
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime, timedelta
import joblib
import json
from pathlib import Path

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Deep Learning (LSTM)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False

from ..utils.logger import setup_logger
from ..core.config import Config

logger = setup_logger(__name__, Config.LOG_FILE, Config.LOG_LEVEL)


class TechnicalIndicators:
    """Calcula indicadores técnicos para análise de mercado"""
    
    @staticmethod
    def calculate_rsi(prices: np.ndarray, period: int = 14) -> float:
        """
        Calcula o RSI (Relative Strength Index)
        
        Args:
            prices: Array de preços
            period: Período para cálculo
            
        Returns:
            Valor do RSI (0-100)
        """
        if len(prices) < period + 1:
            return 50.0
        
        deltas = np.diff(prices)
        gains = np.where(deltas > 0, deltas, 0)
        losses = np.where(deltas < 0, -deltas, 0)
        
        avg_gain = np.mean(gains[-period:])
        avg_loss = np.mean(losses[-period:])
        
        if avg_loss == 0:
            return 100.0
        
        rs = avg_gain / avg_loss
        rsi = 100 - (100 / (1 + rs))
        
        return float(rsi)
    
    @staticmethod
    def calculate_macd(prices: np.ndarray, fast: int = 12, slow: int = 26, signal: int = 9) -> Dict[str, float]:
        """
        Calcula o MACD (Moving Average Convergence Divergence)
        
        Args:
            prices: Array de preços
            fast: Período EMA rápida
            slow: Período EMA lenta
            signal: Período da linha de sinal
            
        Returns:
            Dict com macd, signal e histogram
        """
        if len(prices) < slow:
            return {"macd": 0.0, "signal": 0.0, "histogram": 0.0}
        
        # Calcular EMAs
        ema_fast = TechnicalIndicators._ema(prices, fast)
        ema_slow = TechnicalIndicators._ema(prices, slow)
        
        macd_line = ema_fast - ema_slow
        signal_line = TechnicalIndicators._ema(np.array([macd_line]), signal)
        histogram = macd_line - signal_line
        
        return {
            "macd": float(macd_line),
            "signal": float(signal_line),
            "histogram": float(histogram)
        }
    
    @staticmethod
    def calculate_bollinger_bands(prices: np.ndarray, period: int = 20, std_dev: int = 2) -> Dict[str, float]:
        """
        Calcula as Bandas de Bollinger
        
        Args:
            prices: Array de preços
            period: Período para média móvel
            std_dev: Número de desvios padrão
            
        Returns:
            Dict com upper, middle e lower bands
        """
        if len(prices) < period:
            middle = float(np.mean(prices))
            return {"upper": middle, "middle": middle, "lower": middle}
        
        middle_band = np.mean(prices[-period:])
        std = np.std(prices[-period:])
        
        upper_band = middle_band + (std_dev * std)
        lower_band = middle_band - (std_dev * std)
        
        return {
            "upper": float(upper_band),
            "middle": float(middle_band),
            "lower": float(lower_band)
        }
    
    @staticmethod
    def calculate_moving_averages(prices: np.ndarray) -> Dict[str, float]:
        """
        Calcula médias móveis simples
        
        Args:
            prices: Array de preços
            
        Returns:
            Dict com MA de diferentes períodos
        """
        mas = {}
        periods = [5, 10, 20, 50, 100, 200]
        
        for period in periods:
            if len(prices) >= period:
                mas[f"ma_{period}"] = float(np.mean(prices[-period:]))
            else:
                mas[f"ma_{period}"] = float(np.mean(prices))
        
        return mas
    
    @staticmethod
    def calculate_volume_indicators(volumes: np.ndarray, prices: np.ndarray) -> Dict[str, float]:
        """
        Calcula indicadores baseados em volume
        
        Args:
            volumes: Array de volumes
            prices: Array de preços
            
        Returns:
            Dict com indicadores de volume
        """
        if len(volumes) < 20:
            return {
                "volume_ma": float(np.mean(volumes)),
                "volume_ratio": 1.0,
                "obv": 0.0
            }
        
        volume_ma = np.mean(volumes[-20:])
        current_volume = volumes[-1]
        volume_ratio = current_volume / volume_ma if volume_ma > 0 else 1.0
        
        # OBV (On-Balance Volume)
        obv = 0.0
        for i in range(1, len(prices)):
            if prices[i] > prices[i-1]:
                obv += volumes[i]
            elif prices[i] < prices[i-1]:
                obv -= volumes[i]
        
        return {
            "volume_ma": float(volume_ma),
            "volume_ratio": float(volume_ratio),
            "obv": float(obv)
        }
    
    @staticmethod
    def _ema(prices: np.ndarray, period: int) -> float:
        """Calcula EMA (Exponential Moving Average)"""
        if len(prices) < period:
            return float(np.mean(prices))
        
        multiplier = 2 / (period + 1)
        ema = np.mean(prices[:period])
        
        for price in prices[period:]:
            ema = (price * multiplier) + (ema * (1 - multiplier))
        
        return float(ema)


class AIPredictor:
    """Motor de IA para previsão de mercado usando Machine Learning"""
    
    def __init__(self, model_dir: str = "backend/data/models"):
        """
        Inicializa o preditor de IA
        
        Args:
            model_dir: Diretório para salvar modelos
        """
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)
        
        self.rf_model: Optional[RandomForestClassifier] = None
        self.lstm_model: Optional[Any] = None
        self.scaler: Optional[StandardScaler] = None
        self.feature_names: List[str] = []
        
        self.indicators = TechnicalIndicators()
        self._is_trained = False
        self._last_training = None
        
        logger.info("AIPredictor inicializado")
    
    async def prepare_features(self, candles: List[Dict]) -> Tuple[np.ndarray, List[str]]:
        """
        Prepara features a partir dos candles
        
        Args:
            candles: Lista de candles com OHLCV
            
        Returns:
            Tuple com (features array, feature names)
        """
        if len(candles) < 200:
            logger.warning(f"Poucos candles para análise: {len(candles)}")
        
        # Extrair dados
        closes = np.array([c['close'] for c in candles])
        opens = np.array([c['open'] for c in candles])
        highs = np.array([c['max'] for c in candles])
        lows = np.array([c['min'] for c in candles])
        volumes = np.array([c['volume'] for c in candles])
        
        # Calcular indicadores
        features = {}
        
        # RSI
        features['rsi'] = self.indicators.calculate_rsi(closes)
        
        # MACD
        macd = self.indicators.calculate_macd(closes)
        features.update(macd)
        
        # Bollinger Bands
        bb = self.indicators.calculate_bollinger_bands(closes)
        features['bb_upper'] = bb['upper']
        features['bb_middle'] = bb['middle']
        features['bb_lower'] = bb['lower']
        features['bb_position'] = (closes[-1] - bb['lower']) / (bb['upper'] - bb['lower']) if bb['upper'] != bb['lower'] else 0.5
        
        # Moving Averages
        mas = self.indicators.calculate_moving_averages(closes)
        features.update(mas)
        
        # Volume
        vol_indicators = self.indicators.calculate_volume_indicators(volumes, closes)
        features.update(vol_indicators)
        
        # Price action
        features['price_change'] = (closes[-1] - closes[-2]) / closes[-2] if len(closes) > 1 else 0
        features['price_range'] = (highs[-1] - lows[-1]) / closes[-1] if closes[-1] != 0 else 0
        features['body_size'] = abs(closes[-1] - opens[-1]) / closes[-1] if closes[-1] != 0 else 0
        
        # Momentum
        if len(closes) >= 10:
            features['momentum_10'] = (closes[-1] - closes[-10]) / closes[-10]
        else:
            features['momentum_10'] = 0
        
        # Volatility
        if len(closes) >= 20:
            features['volatility'] = np.std(closes[-20:]) / np.mean(closes[-20:])
        else:
            features['volatility'] = 0
        
        # Trend
        if len(closes) >= 50:
            features['trend_50'] = 1 if closes[-1] > np.mean(closes[-50:]) else -1
        else:
            features['trend_50'] = 0
        
        feature_names = list(features.keys())
        feature_array = np.array([list(features.values())])
        
        return feature_array, feature_names
    
    async def train_random_forest(self, historical_data: List[Dict], labels: np.ndarray) -> Dict[str, float]:
        """
        Treina modelo Random Forest
        
        Args:
            historical_data: Dados históricos de candles
            labels: Labels (1=BUY, 0=HOLD, -1=SELL)
            
        Returns:
            Métricas de performance
        """
        logger.info("Iniciando treinamento do Random Forest...")
        
        # Preparar features
        all_features = []
        for candles in historical_data:
            features, feature_names = await self.prepare_features(candles)
            all_features.append(features[0])
        
        X = np.array(all_features)
        y = labels
        
        # Normalizar
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Treinar modelo
        self.rf_model = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        
        loop = asyncio.get_event_loop()
        await loop.run_in_executor(None, self.rf_model.fit, X_train, y_train)
        
        # Avaliar
        y_pred = self.rf_model.predict(X_test)
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
        }
        
        self.feature_names = feature_names
        self._is_trained = True
        self._last_training = datetime.now()
        
        logger.info(f"Random Forest treinado - Accuracy: {metrics['accuracy']:.4f}")
        
        # Salvar modelo
        await self.save_model('random_forest')
        
        return metrics
    
    async def train_lstm(self, historical_data: List[Dict], labels: np.ndarray, epochs: int = 50) -> Dict[str, float]:
        """
        Treina modelo LSTM
        
        Args:
            historical_data: Dados históricos de candles
            labels: Labels (1=BUY, 0=HOLD, -1=SELL)
            epochs: Número de épocas
            
        Returns:
            Métricas de performance
        """
        if not TENSORFLOW_AVAILABLE:
            logger.warning("TensorFlow não disponível. Usando apenas Random Forest.")
            return {}
        
        logger.info("Iniciando treinamento do LSTM...")
        
        # Preparar sequências
        sequence_length = 60
        all_features = []
        
        for candles in historical_data:
            features, _ = await self.prepare_features(candles)
            all_features.append(features[0])
        
        X = np.array(all_features)
        
        # Normalizar
        if self.scaler is None:
            self.scaler = StandardScaler()
            X_scaled = self.scaler.fit_transform(X)
        else:
            X_scaled = self.scaler.transform(X)
        
        # Criar sequências
        X_seq = []
        y_seq = []
        
        for i in range(sequence_length, len(X_scaled)):
            X_seq.append(X_scaled[i-sequence_length:i])
            y_seq.append(labels[i])
        
        X_seq = np.array(X_seq)
        y_seq = np.array(y_seq)
        
        # Converter labels para one-hot
        y_seq_categorical = keras.utils.to_categorical(y_seq + 1, num_classes=3)  # -1,0,1 -> 0,1,2
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_seq, y_seq_categorical, test_size=0.2, random_state=42
        )
        
        # Construir modelo
        self.lstm_model = Sequential([
            LSTM(128, return_sequences=True, input_shape=(sequence_length, X.shape[1])),
            Dropout(0.3),
            LSTM(64, return_sequences=True),
            Dropout(0.3),
            LSTM(32),
            Dropout(0.2),
            Dense(64, activation='relu'),
            Dropout(0.2),
            Dense(3, activation='softmax')
        ])
        
        self.lstm_model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Treinar
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        loop = asyncio.get_event_loop()
        history = await loop.run_in_executor(
            None,
            lambda: self.lstm_model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=32,
                validation_split=0.2,
                callbacks=[early_stop],
                verbose=0
            )
        )
        
        # Avaliar
        y_pred = self.lstm_model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_test_classes = np.argmax(y_test, axis=1)
        
        metrics = {
            'accuracy': accuracy_score(y_test_classes, y_pred_classes),
            'precision': precision_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0),
            'recall': recall_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0),
            'f1': f1_score(y_test_classes, y_pred_classes, average='weighted', zero_division=0)
        }
        
        logger.info(f"LSTM treinado - Accuracy: {metrics['accuracy']:.4f}")
        
        # Salvar modelo
        await self.save_model('lstm')
        
        return metrics
    
    async def predict(self, candles: List[Dict], use_lstm: bool = False) -> Dict[str, Any]:
        """
        Gera previsão de mercado
        
        Args:
            candles: Lista de candles recentes
            use_lstm: Se True, usa LSTM; caso contrário, Random Forest
            
        Returns:
            Dict com signal, confidence e indicators
        """
        if not self._is_trained:
            logger.warning("Modelo não treinado. Carregando modelo salvo...")
            await self.load_model('random_forest')
        
        # Preparar features
        features, feature_names = await self.prepare_features(candles)
        
        # Normalizar
        if self.scaler:
            features_scaled = self.scaler.transform(features)
        else:
            features_scaled = features
        
        # Prever
        if use_lstm and self.lstm_model and TENSORFLOW_AVAILABLE:
            # LSTM precisa de sequência
            prediction_proba = self.lstm_model.predict(features_scaled.reshape(1, 1, -1), verbose=0)[0]
            prediction = np.argmax(prediction_proba) - 1  # 0,1,2 -> -1,0,1
            confidence = float(np.max(prediction_proba))
        elif self.rf_model:
            prediction = self.rf_model.predict(features_scaled)[0]
            prediction_proba = self.rf_model.predict_proba(features_scaled)[0]
            confidence = float(np.max(prediction_proba))
        else:
            logger.error("Nenhum modelo disponível")
            return {
                "signal": "HOLD",
                "confidence": 0.0,
                "indicators": {}
            }
        
        # Mapear sinal
        signal_map = {-1: "SELL", 0: "HOLD", 1: "BUY"}
        signal = signal_map.get(int(prediction), "HOLD")
        
        # Preparar indicadores
        indicators = dict(zip(feature_names, features[0]))
        
        result = {
            "signal": signal,
            "confidence": confidence,
            "indicators": indicators,
            "timestamp": datetime.now().isoformat(),
            "model_used": "LSTM" if use_lstm else "RandomForest"
        }
        
        logger.info(f"Previsão: {signal} (confiança: {confidence:.2%})")
        
        return result
    
    async def save_model(self, model_type: str = 'random_forest') -> None:
        """Salva modelo treinado"""
        try:
            if model_type == 'random_forest' and self.rf_model:
                model_path = self.model_dir / 'rf_model.joblib'
                scaler_path = self.model_dir / 'scaler.joblib'
                
                joblib.dump(self.rf_model, model_path)
                joblib.dump(self.scaler, scaler_path)
                
                # Salvar metadados
                metadata = {
                    'feature_names': self.feature_names,
                    'last_training': self._last_training.isoformat() if self._last_training else None,
                    'model_type': 'RandomForest'
                }
                
                with open(self.model_dir / 'metadata.json', 'w') as f:
                    json.dump(metadata, f, indent=2)
                
                logger.info(f"Modelo Random Forest salvo em {model_path}")
            
            elif model_type == 'lstm' and self.lstm_model and TENSORFLOW_AVAILABLE:
                model_path = self.model_dir / 'lstm_model.h5'
                self.lstm_model.save(model_path)
                logger.info(f"Modelo LSTM salvo em {model_path}")
        
        except Exception as e:
            logger.error(f"Erro ao salvar modelo: {str(e)}")
    
    async def load_model(self, model_type: str = 'random_forest') -> bool:
        """Carrega modelo salvo"""
        try:
            if model_type == 'random_forest':
                model_path = self.model_dir / 'rf_model.joblib'
                scaler_path = self.model_dir / 'scaler.joblib'
                metadata_path = self.model_dir / 'metadata.json'
                
                if not model_path.exists():
                    logger.warning("Modelo Random Forest não encontrado")
                    return False
                
                self.rf_model = joblib.load(model_path)
                self.scaler = joblib.load(scaler_path)
                
                if metadata_path.exists():
                    with open(metadata_path, 'r') as f:
                        metadata = json.load(f)
                        self.feature_names = metadata.get('feature_names', [])
                        self._last_training = datetime.fromisoformat(metadata['last_training']) if metadata.get('last_training') else None
                
                self._is_trained = True
                logger.info("Modelo Random Forest carregado com sucesso")
                return True
            
            elif model_type == 'lstm' and TENSORFLOW_AVAILABLE:
                model_path = self.model_dir / 'lstm_model.h5'
                
                if not model_path.exists():
                    logger.warning("Modelo LSTM não encontrado")
                    return False
                
                self.lstm_model = load_model(model_path)
                logger.info("Modelo LSTM carregado com sucesso")
                return True
        
        except Exception as e:
            logger.error(f"Erro ao carregar modelo: {str(e)}")
            return False
    
    async def should_retrain(self, days_threshold: int = 7) -> bool:
        """
        Verifica se modelo deve ser retreinado
        
        Args:
            days_threshold: Dias desde último treinamento
            
        Returns:
            True se deve retreinar
        """
        if not self._last_training:
            return True
        
        days_since_training = (datetime.now() - self._last_training).days
        return days_since_training >= days_threshold
    
    @property
    def is_trained(self) -> bool:
        """Verifica se modelo está treinado"""
        return self._is_trained
'''

# Initialize Editor and write the AI predictor module
editor = Editor()
await editor.write('/workspace/backend/services/ai_predictor.py', ai_predictor_code)
print("✓ AI Predictor module created at /workspace/backend/services/ai_predictor.py")

✓ AI Predictor module created at /workspace/backend/services/ai_predictor.py


2026-01-01 19:39:30.534 | INFO     | metagpt.const:get_metagpt_root:33 - PROJECT_ROOT set from environment variable to /
